# AI Escape Room — Setup

Run these cells from top to bottom. In Google Colab, add your Gemini API key in **Secrets** using the name `GEMINI_API_KEY`, and enable notebook access to that secret.

In [ ]:
!pip install -q "google-genai>=2.12.1,<3" gradio

In [ ]:
import google.genai
import google.auth
import gradio

print("Gemini SDK:", google.genai.__version__)
print("Google Auth:", google.auth.__version__)
print("Gradio:", gradio.__version__)

In [ ]:
from google.colab import userdata
from google import genai

# Retrieve API key from Colab Secrets
api_key = userdata.get("GEMINI_API_KEY")

# Connect to Gemini
client = genai.Client(api_key=api_key)

print("Gemini API connected successfully!")

## Version 1 — LLM-managed escape room

In this implementation, Gemini handles the game progression through the conversation. Python does not maintain or validate a separate puzzle/inventory state.

In [ ]:
game_master_prompt = """
You are the Game Master of an interactive AI escape room called
'The Forgotten Library'.

YOUR ROLE:
- Act as an immersive, mysterious and friendly Game Master.
- Describe the library and its objects.
- Respond naturally to the player's actions.
- Give hints when the player is genuinely stuck.
- Maintain continuity using the conversation history.

GAME ENVIRONMENT:

The player wakes up inside a locked library.

There are exactly FIVE interactive objects:

1. Bookshelf
2. Painting
3. Desk
4. Locked Drawer
5. Exit Door

PUZZLES:

Puzzle 1 - Bookshelf:
The player discovers unusual books containing hidden numbers.

Puzzle 2 - Painting:
The painting contains a hidden message that helps interpret the numbers.

Puzzle 3 - Desk:
A handwritten note provides the final clue needed to determine the four-digit combination.

FINAL ESCAPE:
The correct combination is 2749.
The locked drawer contains the key to the exit door.
The player must investigate the clues, unlock the drawer,
collect the key and use it to escape.

GAME RULES:

- Only describe objects that exist in the library.
- Do not introduce additional interactive objects.
- Encourage exploration.
- Do not immediately reveal all puzzle answers.
- Give clues progressively.
- Remember the player's previous messages.
- Do not declare the player escaped before the final challenge.
- If the player asks for a hint, provide a useful but gradual hint.
- Keep responses engaging and relatively short.

IMPORTANT:
This is Version 1. You are responsible for managing the
game progression through the conversation.

Begin by welcoming the player and describing the library.
Ask what they would like to investigate first.
"""

In [ ]:
import gradio as gr

def play_version1(message, history, chat):
    if not message or not message.strip():
        return history or [], chat

    if chat is None:
        chat = client.chats.create(
            model="gemini-3.5-flash-lite",
            config={
                "system_instruction": game_master_prompt,
                "temperature": 0.7
            }
        )

    response = chat.send_message(message)
    history = history or []
    history = history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": response.text}
    ]
    return history, chat

def restart_version1():
    return [], None

with gr.Blocks(title="AI Escape Room — Version 1") as demo_v1:
    gr.Markdown("""
    # 🔐 The Forgotten Library — Version 1
    **LLM-managed game:** Gemini manages the story, puzzle progression,
    and decisions through conversation. There is no separate Python
    game-state validator in this version.
    """)
    chatbot = gr.Chatbot(label="Escape Room Adventure", height=450)
    message = gr.Textbox(
        placeholder="What would you like to investigate?",
        label="Your message"
    )
    with gr.Row():
        send = gr.Button("Send Message", variant="primary")
        restart = gr.Button("Restart Game")
    chat_state = gr.State(None)

    send.click(
        play_version1,
        inputs=[message, chatbot, chat_state],
        outputs=[chatbot, chat_state]
    ).then(lambda: "", outputs=message)

    message.submit(
        play_version1,
        inputs=[message, chatbot, chat_state],
        outputs=[chatbot, chat_state]
    ).then(lambda: "", outputs=message)

    restart.click(
        restart_version1,
        outputs=[chatbot, chat_state]
    )

demo_v1.queue()
demo_v1.launch(share=True, debug=True, theme=gr.themes.Soft())
